In [2]:

import numpy as np
from scipy.linalg import eigh
from scipy.optimize import minimize

# 1. Duas marginais
rho_A = np.diag([0.7, 0.3]).astype(complex)
rho_B = np.diag([0.6, 0.4]).astype(complex)

epsilon = 0.5
H = np.zeros((4, 4), dtype=complex)

# 2. Base de Pauli
I = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], complex)
sy = np.array([[0, -1j], [1j, 0]], complex)
sz = np.diag([1, -1]).astype(complex)

paulis = [sx, sy, sz]

# Restrições Tr(Q_i Gamma) = q_i
Q = [np.eye(4, dtype=complex)]
q = [1.0]

for sigma in paulis:
    Q.append(np.kron(sigma, I))
    q.append(np.trace(sigma @ rho_A).real)

for sigma in paulis:
    Q.append(np.kron(I, sigma))
    q.append(np.trace(sigma @ rho_B).real)

Q = np.array(Q)
q = np.array(q)

# 3. Dual entrópico

def avaliar_dual(u):
    Q_u = np.einsum("n,nij->ij", u, Q)
    K = (Q_u - H) / epsilon

    autovalores, V = eigh(K)
    Gamma = (V * np.exp(autovalores)) @ V.conj().T

    # Função dual
    D = np.dot(u, q) - epsilon * np.trace(Gamma).real

    # Valores esperados Tr(Q_n Gamma)
    valores = np.einsum("nij,ji->n", Q, Gamma).real

    # Gradiente analítico
    grad_D = q - valores

    return -D, -grad_D



# 4. Maximizar o dual
resultado = minimize(
    avaliar_dual,
    x0=np.zeros(len(q)),
    jac=True,
    method="L-BFGS-B",
    options={"gtol": 1e-12, "maxiter": 2000}
)

# 5. Reconstruir o operador global
u = resultado.x
K = np.einsum("i,ijk->jk", u, Q) / epsilon

autovalores, V = eigh(K)
Gamma = (V * np.exp(autovalores)) @ V.conj().T

# 6. Comparar com a solução exata
Gamma_exata = np.kron(rho_A, rho_B)

print("Convergência:", resultado.success)
print("Estado reconstruído:")
print(np.round(Gamma, 6))

print("\nSolução exata:")
print(Gamma_exata)

print("\nErro Frobenius:",
      np.linalg.norm(Gamma - Gamma_exata))

print("Resíduo das restrições:",
      np.linalg.norm(avaliar_dual(u)[1]))


Convergência: True
Estado reconstruído:
[[0.420001+0.j 0.      +0.j 0.      +0.j 0.      +0.j]
 [0.      +0.j 0.28    +0.j 0.      +0.j 0.      +0.j]
 [0.      +0.j 0.      +0.j 0.18    +0.j 0.      +0.j]
 [0.      +0.j 0.      +0.j 0.      +0.j 0.12    +0.j]]

Solução exata:
[[0.42+0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.28+0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.18+0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.12+0.j]]

Erro Frobenius: 5.607771936028383e-07
Resíduo das restrições: 1.0868231142326161e-06
